# Inspection IK Trace Plot

Select an IK trace folder or file, then choose an iteration range to inspect joint convergence and error curves.

In [1]:
from pathlib import Path
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'python').exists() and (candidate / 'experiment').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
DEFAULT_ROOT = REPO_ROOT / 'experiment' / 'inspection_ik'
print('default root:', DEFAULT_ROOT)

default root: d:\flame_robotics_drt\experiment\experiment\inspection_ik


In [ ]:
def find_traces(root):
    root = Path(root).expanduser().resolve()
    if root.is_file():
        return [root]
    json_paths = sorted(root.rglob('inspection_ik_*.json'))
    json_stems = {p.with_suffix('').name for p in json_paths}
    csv_paths = [
        p for p in sorted(root.rglob('inspection_ik_*.csv'))
        if p.with_suffix('').name not in json_stems
    ]
    return json_paths + csv_paths

def resolve_csv(path):
    path = Path(path).resolve()
    meta = {}
    if path.suffix.lower() == '.json':
        meta = json.loads(path.read_text(encoding='utf-8'))
        csv_path = Path(meta['csv_path'])
        if not csv_path.is_absolute():
            candidates = [
                (REPO_ROOT / csv_path).resolve(),
                (Path.cwd() / csv_path).resolve(),
                (path.parent / csv_path).resolve(),
            ]
            csv_path = next((candidate for candidate in candidates if candidate.exists()), candidates[0])
        return csv_path, meta
    paired = path.with_suffix('.json')
    if paired.exists():
        meta = json.loads(paired.read_text(encoding='utf-8'))
    return path, meta

def load_trace(path):
    csv_path, meta = resolve_csv(path)
    rows = []
    with csv_path.open('r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fields = reader.fieldnames or []
        q_fields = [name for name in fields if name.startswith('q')]
        q_fields.sort(key=lambda name: int(name.split('_', 1)[0][1:]))
        joint_names = [name.split('_', 1)[1] for name in q_fields]
        for row in reader:
            rows.append({
                'iteration': int(float(row['iteration'])),
                'err_norm': float(row['err_norm']),
                'position_error': float(row['position_error']),
                'orientation_error': float(row['orientation_error']),
                'q': np.array([float(row[name]) for name in q_fields], dtype=float),
            })
    return csv_path, meta, joint_names, rows

def normalize_q(q_values):
    q_values = np.asarray(q_values, dtype=float)
    q_min = np.nanmin(q_values, axis=0)
    q_max = np.nanmax(q_values, axis=0)
    span = q_max - q_min
    span[span < 1e-12] = 1.0
    return (q_values - q_min) / span

In [ ]:
root_text = widgets.Text(value=str(DEFAULT_ROOT), description='root/file:', layout=widgets.Layout(width='900px'))
reload_button = widgets.Button(description='Reload traces')
trace_dropdown = widgets.Dropdown(description='trace:', layout=widgets.Layout(width='900px'))
range_slider = widgets.IntRangeSlider(description='range:', min=0, max=1, value=(0, 1), continuous_update=False, layout=widgets.Layout(width='900px'))
normalized_check = widgets.Checkbox(value=False, description='min-max normalize joint values for plot only')
plot_button = widgets.Button(description='Plot selected range')
out = widgets.Output()

trace_paths = []

def reload_traces(_=None):
    global trace_paths
    trace_paths = find_traces(root_text.value)
    options = [(str(p.relative_to(Path.cwd())) if p.is_relative_to(Path.cwd()) else str(p), str(p)) for p in trace_paths]
    trace_dropdown.options = options
    with out:
        clear_output(wait=True)
        print(f'found {len(trace_paths)} trace(s)')

def update_range(*_):
    if not trace_dropdown.value:
        return
    _, _, _, rows = load_trace(trace_dropdown.value)
    iters = [row['iteration'] for row in rows]
    range_slider.min = min(iters)
    range_slider.max = max(iters)
    range_slider.value = (min(iters), max(iters))

def plot_selected(_=None):
    if not trace_dropdown.value:
        return
    csv_path, meta, joint_names, rows = load_trace(trace_dropdown.value)
    lo, hi = range_slider.value
    rows = [row for row in rows if lo <= row['iteration'] <= hi]
    if not rows:
        return
    iterations = np.array([row['iteration'] for row in rows])
    q_values = np.vstack([row['q'] for row in rows])
    plot_q = normalize_q(q_values) if normalized_check.value else q_values
    ik = meta.get('ik_result', {}) if isinstance(meta, dict) else {}
    title = f"{csv_path.name} | solver={ik.get('solver', '-')} normalize={ik.get('normalize', '-')}"
    with out:
        clear_output(wait=True)
        fig, (ax_q, ax_err) = plt.subplots(2, 1, figsize=(14, max(7, 0.45 * len(joint_names) + 4)), sharex=True, gridspec_kw={'height_ratios': [3, 1.5]})
        for idx, name in enumerate(joint_names):
            ax_q.plot(iterations, plot_q[:, idx], label=f'q{idx}: {name}', linewidth=1.5)
        ax_q.set_title(title)
        ax_q.set_ylabel('joint value' + (' (plot-normalized)' if normalized_check.value else ''))
        ax_q.grid(True, alpha=0.25)
        ax_q.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
        ax_err.plot(iterations, [row['err_norm'] for row in rows], label='SE3 err norm')
        ax_err.plot(iterations, [row['position_error'] for row in rows], label='position error [m]')
        ax_err.plot(iterations, [row['orientation_error'] for row in rows], label='orientation error [rad]')
        ax_err.set_xlabel('iteration')
        ax_err.set_ylabel('error')
        ax_err.grid(True, alpha=0.25)
        ax_err.legend(loc='best')
        fig.tight_layout()
        plt.show()
        print('csv:', csv_path)
        print('range:', (lo, hi), 'rows:', len(rows))

reload_button.on_click(reload_traces)
trace_dropdown.observe(update_range, names='value')
plot_button.on_click(plot_selected)
display(widgets.VBox([root_text, reload_button, trace_dropdown, range_slider, normalized_check, plot_button, out]))
reload_traces()

## Angle Plot And Mesh Replay

Use the selected trace and iteration range above. Revolute joints are shown in degrees, while track/carriage/linear/prismatic joints are shown separately in raw units.

In [ ]:
import subprocess
import sys

angle_button = widgets.Button(description='Plot angle range')
mesh_step = widgets.IntText(value=1, description='mesh step:')
mesh_delay = widgets.FloatText(value=0.03, description='delay:')
mesh_command_button = widgets.Button(description='Print mesh command')
mesh_launch_button = widgets.Button(description='Launch mesh replay')
angle_out = widgets.Output()
mesh_out = widgets.Output()

def is_prismatic_name(name):
    lower = str(name).lower()
    return any(key in lower for key in ['linear', 'track', 'carriage', 'prismatic', 'slide'])

def plot_angle_selected(_=None):
    if not trace_dropdown.value:
        return
    csv_path, meta, joint_names, rows = load_trace(trace_dropdown.value)
    lo, hi = range_slider.value
    rows = [row for row in rows if lo <= row['iteration'] <= hi]
    if not rows:
        return
    iterations = np.array([row['iteration'] for row in rows])
    q_values = np.vstack([row['q'] for row in rows])
    revolute_idx = [i for i, name in enumerate(joint_names) if not is_prismatic_name(name)]
    prismatic_idx = [i for i, name in enumerate(joint_names) if is_prismatic_name(name)]
    with angle_out:
        clear_output(wait=True)
        nplots = 2 if prismatic_idx else 1
        fig, axes = plt.subplots(nplots, 1, figsize=(14, 5 if nplots == 1 else 8), sharex=True)
        axes = np.atleast_1d(axes)
        ax_angle = axes[0]
        for idx in revolute_idx:
            ax_angle.plot(iterations, np.rad2deg(q_values[:, idx]), label=f'q{idx}: {joint_names[idx]}', linewidth=1.5)
        ax_angle.set_title(f'Joint angle convergence | {csv_path.name}')
        ax_angle.set_ylabel('angle [deg]')
        ax_angle.grid(True, alpha=0.25)
        ax_angle.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
        if prismatic_idx:
            ax_lin = axes[1]
            for idx in prismatic_idx:
                ax_lin.plot(iterations, q_values[:, idx], label=f'q{idx}: {joint_names[idx]}', linewidth=1.5)
            ax_lin.set_ylabel('linear joint value')
            ax_lin.grid(True, alpha=0.25)
            ax_lin.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8)
        axes[-1].set_xlabel('iteration')
        fig.tight_layout()
        plt.show()
        print('csv:', csv_path)
        print('range:', (lo, hi), 'rows:', len(rows))

def mesh_replay_command():
    if not trace_dropdown.value:
        return None
    lo, hi = range_slider.value
    trace = Path(trace_dropdown.value).resolve()
    script = (REPO_ROOT / 'experiment' / 'inspection_ik_visualize.py').resolve()
    return [
        sys.executable,
        str(script),
        str(trace),
        '--start-index', str(lo),
        '--end-index', str(hi),
        '--step', str(max(1, int(mesh_step.value))),
        '--delay', str(max(0.0, float(mesh_delay.value))),
    ]

def print_mesh_command(_=None):
    cmd = mesh_replay_command()
    with mesh_out:
        clear_output(wait=True)
        if cmd is None:
            print('select a trace first')
            return
        print(' '.join(f'"{part}"' if ' ' in part else part for part in cmd))

def launch_mesh_replay(_=None):
    cmd = mesh_replay_command()
    with mesh_out:
        clear_output(wait=True)
        if cmd is None:
            print('select a trace first')
            return
        print('launching:')
        print(' '.join(f'"{part}"' if ' ' in part else part for part in cmd))
        subprocess.Popen(cmd, cwd=str(REPO_ROOT))

angle_button.on_click(plot_angle_selected)
mesh_command_button.on_click(print_mesh_command)
mesh_launch_button.on_click(launch_mesh_replay)
display(widgets.VBox([
    angle_button,
    angle_out,
    widgets.HBox([mesh_step, mesh_delay, mesh_command_button, mesh_launch_button]),
    mesh_out,
]))